In [1]:
import os
print(os.getcwd())

/Users/rohanjairam/projects/conversational-analytics-dashboard


In [2]:
from src.llm_engine import generate_code_from_query
from src.query_executor import run_generated_code
import pandas as pd

In [3]:
df = pd.read_csv("/Users/rohanjairam/projects/conversational-analytics-dashboard/data/ecommerce.csv", encoding="latin1")
columns = list(df.columns)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:
columns = df.columns.tolist()

In [9]:
query = "What are the top 3 countries by total sales?"

In [10]:
code = generate_code_from_query(query, columns)

# Print for inspection
print("🧠 Generated Code:\n", code)

🧠 Generated Code:
 ```python
# First, create a new column for total sales by multiplying Quantity and UnitPrice
df['TotalSales'] = df['Quantity'] * df['UnitPrice']

# Then, group by Country and sum the TotalSales, sort in descending order and take the top 3
result_df = df.groupby('Country')['TotalSales'].sum().sort_values(ascending=False).head(3).reset_index()
```


In [11]:
# If sometimes the model returns markdown formatting (```python ... ```)
# clean it up
if code.startswith("```"):
    code = code.strip("```").replace("python", "").strip()

print("🧹 Cleaned Code:\n", code)

🧹 Cleaned Code:
 # First, create a new column for total sales by multiplying Quantity and UnitPrice
df['TotalSales'] = df['Quantity'] * df['UnitPrice']

# Then, group by Country and sum the TotalSales, sort in descending order and take the top 3
result_df = df.groupby('Country')['TotalSales'].sum().sort_values(ascending=False).head(3).reset_index()


In [12]:
local_vars = {"df": df.copy()}

try:
    exec(code, globals(), local_vars)
    result_df = local_vars.get("result_df", None)
    
    if result_df is not None:
        display(result_df.head(10))
    else:
        print("⚠️ No DataFrame called `result_df` was created.")
except Exception as e:
    print("❌ Error while executing code:\n", e)

,Country,TotalSales
0,United Kingdom,8187806.364
1,Netherlands,284661.540
2,EIRE,263276.820
